# Arquitectura y modelo — SaaS Metrics (proyecto ficticio, Reto Alegra)

**Nota:** todos los datos de este proyecto son sintéticos (generados con `Faker`,
semilla fija). Ninguna cifra corresponde a Alegra ni a un cliente real. El objetivo es
demostrar cómo abordaría, con ayuda de Claude Code, el problema que describe la
vacante *Business Intelligence & Analytics Partner*: convertir datos dispersos de CRM,
facturación y contratos en métricas SaaS confiables (ARR/MRR, NRR, CAC, LTV, churn).

## El problema

Un SaaS ficticio factura desde **Stripe**, gestiona su embudo comercial en un **CRM**
estilo HubSpot y firma **contratos** formales para sus cuentas grandes. Los tres
sistemas no comparten un identificador común: la misma empresa puede tener un ID de
CRM, un ID de cliente en Billing y ningún ID en absoluto en Contratos (solo la razón
social escrita a mano). La pregunta de negocio — *¿cuál es el MRR real, y por qué no
cuadra entre sistemas?* — no se puede responder sin antes resolver esa identidad.

## Arquitectura en 4 capas

```
                 ORIGENES HETEROGENEOS
   CRM (HubSpot)   Billing (Stripe)   Contratos (Excel)   Marketing (canal x mes)
        |                |                   |                    |
        +----------------+-------------------+--------------------+
                                  v
 [ RAW ]      copia inmutable de cada export, tal como llega
                                  |
                                  v
 [ BRONZE ]   bronze_crm / bronze_billing / bronze_contrato / bronze_marketing
              ingesta literal: TODO texto, sin castear, con linaje (id_carga, hash_fila)
                                  |
                                  v
 [ SILVER ]   dim_cliente / dim_plan / dim_canal / dim_periodo
              fact_evento_suscripcion / fact_contrato / fact_gasto_adquisicion
              map_alias_cliente / map_alias_plan / map_alias_canal
              aqui se resuelve identidad, alias y formato — ver mas abajo
                                  |
                                  v
 [ GOLD ]     gold_v_* — vistas de consumo. Power BI y los notebooks leen SOLO de aqui.
```

Las capas están separadas para que un origen nuevo (un cuarto sistema, un país nuevo) solo
agregue un lector hacia Bronze, sin tocar Silver ni Gold.

## El modelo (capa Silver)

**Dimensiones:** `dim_cliente`, `dim_plan`, `dim_canal`, `dim_periodo`, `dim_tasa_cambio`.
**Puentes de alias:** `map_alias_cliente`, `map_alias_plan`, `map_alias_canal` — no se
siembran a mano, las construye el ETL resolviendo las variantes de texto reales que
trae cada archivo.
**Hechos:** `fact_evento_suscripcion` (grano: un evento de alta/upgrade/downgrade/baja/
reactivación), `fact_contrato` (grano: un contrato firmado), `fact_gasto_adquisicion`
(grano: canal × mes).
**Control y calidad:** `ctl_carga` (una fila por archivo procesado, con linaje),
`err_registro_rechazado` (cuarentena: una fila que no se puede tipificar no rompe la
carga, se aísla).

## La decisión de diseño que importa: cómo se resuelve la identidad del cliente

**CRM es la fuente de identidad.** `id_contacto_crm` es estable aunque el nombre de la
empresa cambie (una empresa que se renombra a mitad de año sigue siendo el mismo
contacto en el CRM). Billing y Contratos se resuelven **contra** ese diccionario, nunca
al revés.

**Billing se ancla por `id_customer_stripe`, no por nombre fila a fila.** Esto no es
un capricho: al construir el generador de datos sintéticos, **Faker generó por azar el
mismo nombre de empresa para dos clientes distintos** tres veces ("Rivera Group",
"Castillo LLC", "Gomez-Quintero"). Si la identidad se resolviera solo por nombre
normalizado, esas dos empresas reales se fusionarían por error bajo un solo cliente —
exactamente el tipo de bug silencioso que un pipeline de producción no puede permitirse.
La solución: anclar por el identificador estable de cada sistema (igual que CRM se
ancla por `id_contacto_crm`), y cuando un nombre resulta **ambiguo** (reclamado por más
de una empresa real), neutralizarlo — no asignarlo arbitrariamente al primero que
apareció. Ese cliente queda marcado como huérfano, con nota, para revisión manual.

**Contratos con errores de digitación severos también quedan como huérfanos.** No todo
error de texto es corregible por normalización; forzar un match ahí sería peor que no
tener el dato.

In [1]:
import re
from pathlib import Path

ddl = Path(r"D:\Claudia\Alegra\sql\01_ddl_sqlite.sql").read_text(encoding="utf-8")
tablas = re.findall(r"CREATE TABLE (\w+)", ddl)
print(f"{len(tablas)} tablas definidas en sql/01_ddl_sqlite.sql:\n")
for t in tablas:
    print(" -", t)

17 tablas definidas en sql/01_ddl_sqlite.sql:

 - dim_cliente
 - dim_periodo
 - dim_plan
 - dim_canal
 - dim_tasa_cambio
 - map_alias_cliente
 - map_alias_plan
 - map_alias_canal
 - ctl_carga
 - bronze_crm
 - bronze_billing
 - bronze_contrato
 - bronze_marketing
 - fact_evento_suscripcion
 - fact_contrato
 - fact_gasto_adquisicion
 - err_registro_rechazado
